# AI Gateway dynamic routes with LangChain

A dynamic route is a named flow configured in AI Gateway. The route can choose a model or fall back to another one without changing application code. This notebook calls an existing route through `ChatCloudflareWorkersAI` and inspects the requested route and serving model.

Run `make notebook_sync` and `make notebook_lab` from the repository root. Set `CF_ACCOUNT_ID`, `CF_AI_API_TOKEN`, `AI_GATEWAY`, and `AI_GATEWAY_DYNAMIC_ROUTE` (the route name without `dynamic/`) in the repo-root `.env`. Create and deploy the route in that gateway before running the notebook.

[Cloudflare dynamic route setup](https://developers.cloudflare.com/ai-gateway/features/dynamic-routing/) · [Using a dynamic route](https://developers.cloudflare.com/ai-gateway/features/dynamic-routing/usage/)

In [ ]:
import os

from dotenv import load_dotenv
from langchain_cloudflare import ChatCloudflareWorkersAI

load_dotenv(".env")
account_id = os.getenv("CF_ACCOUNT_ID")
api_token = os.getenv("CF_AI_API_TOKEN")
gateway_id = os.getenv("AI_GATEWAY")
route_name = os.getenv("AI_GATEWAY_DYNAMIC_ROUTE")
missing = [
    name
    for name, value in {
        "CF_ACCOUNT_ID": account_id,
        "CF_AI_API_TOKEN": api_token,
        "AI_GATEWAY": gateway_id,
        "AI_GATEWAY_DYNAMIC_ROUTE": route_name,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(
        f"Set these variables before running the notebook: {', '.join(missing)}"
    )

## Call the route

The model ID is `dynamic/<route name>`, and `ai_gateway` must identify the gateway that owns it. REST requests to dynamic routes require `endpoint_format="openai_compatible"`. The route configuration determines which model actually serves the request.

In [ ]:
route = ChatCloudflareWorkersAI(
    model=f"dynamic/{route_name}",
    account_id=account_id,
    api_token=api_token,
    ai_gateway=gateway_id,
    endpoint_format="openai_compatible",
)
reply = route.invoke("Say hello briefly.")
print(reply.content)
print(
    {
        "requested_route": reply.response_metadata.get("requested_model"),
        "serving_model": reply.response_metadata.get("model_name"),
    }
)

`requested_model` retains the route name. `model_name` is the model reported by the response; it may be `None` if the provider does not report one. Route conditions and fallbacks can cause different calls to use different models.

## Python Worker binding

A Jupyter kernel has no Worker `env.AI` binding. In a Python Worker, pass the same route and gateway to the library and invoke it asynchronously:

```python
route = ChatCloudflareWorkersAI(
    model=f"dynamic/{route_name}",
    binding=self.env.AI,
    ai_gateway=gateway_id,
)
reply = await route.ainvoke("Say hello briefly.")
print(reply.response_metadata)
```